In [1]:

from pathlib import Path
import zipfile
import shutil
import os

# ============================================================
# RESTORE ADT BACKUP — PRESERVE EXISTING FILES
# ============================================================

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "ADT_CV" / "ADT"

# Optional: put your exact backup path here when you
# have more than one attached backup.
BACKUP_PATH = None

def is_adt_zip(path):
    try:
        with zipfile.ZipFile(path) as zf:
            return "ADT_CV/ADT/run.py" in zf.namelist()
    except (OSError, zipfile.BadZipFile):
        return False

if BACKUP_PATH:
    source = Path(BACKUP_PATH)
    assert source.exists(), f"Backup not found: {source}"
else:
    candidates = []

    # ZIP archives containing the complete repository.
    for p in INPUT.rglob("*.zip"):
        if is_adt_zip(p):
            candidates.append(p)

    # Kaggle may already have extracted the ZIP.
    for manifest in INPUT.rglob("BACKUP_MANIFEST.json"):
        folder = manifest.parent
        if (folder / "ADT_CV/ADT/run.py").is_file():
            candidates.append(folder)

    if len(candidates) > 1:
        print("Multiple ADT backups detected:")
        for p in candidates:
            print(" ", p)
        raise RuntimeError(
            "Set BACKUP_PATH to the exact backup "
            "you want to restore."
        )

    if len(candidates) == 0:
        if (REPO / "run.py").is_file():
            print("ADT repository already present.")
            print("No attached complete ZIP was found.")
            print("Existing files left unchanged.")
            source = None
        else:
            raise FileNotFoundError(
                "No ADT backup found. "
                "Attach your ADT backup under Add Input."
            )
    else:
        source = candidates[0]

restored = 0
skipped = 0

def safe_destination(name):
    relative = Path(name)
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Unsafe backup path: {name}")

    target = WORK / relative
    if not target.resolve().is_relative_to(WORK.resolve()):
        raise ValueError(f"Unsafe backup path: {name}")

    return target

def copy_if_missing(name, stream):
    global restored, skipped

    destination = safe_destination(name)

    if destination.exists():
        skipped += 1
        return

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(
        destination.name + ".restoring"
    )

    try:
        with temporary.open("wb") as out:
            shutil.copyfileobj(stream, out)
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)

    restored += 1

if source is not None:
    print("Restoring from:", source)

    if source.is_file():
        with zipfile.ZipFile(source) as zf:
            for item in zf.infolist():
                if item.is_dir():
                    continue
                with zf.open(item) as stream:
                    copy_if_missing(item.filename, stream)
    else:
        for path in source.rglob("*"):
            if not path.is_file() or path.is_symlink():
                continue

            relative = path.relative_to(source)
            with path.open("rb") as stream:
                copy_if_missing(str(relative), stream)

print("\n" + "=" * 65)
print("ADT RESTORE STATUS")
print("=" * 65)
print("Files restored:", restored)
print("Existing files preserved:", skipped)
print("Repository:", REPO)

assert (REPO / "run.py").is_file()
assert (REPO / "io/prepare_vogl_cv.py").is_file()

print("\nRBMA13 learned-PCEN checkpoint status:")

for fold in (1, 2, 3):
    folder = (
        REPO / "experiments/vogl_cnn_b/trainable_pcen"
        / "Architecture/rbma13_trainable_pcen"
        / "RBMA13"
        / f"fold_{fold}"
    )

    complete = all(
        (folder / name).is_file()
        for name in (
            "model.pt",
            "pcen.pt",
            "experiment_config.json",
            "test_results.json"
        )
    )

    print(f"Fold {fold}: {'COMPLETE' if complete else 'NOT COMPLETE'}")

print("\nRESTORE CHECK FINISHED")
print("Existing experiments were not overwritten.")


Restoring from: /kaggle/input/datasets/wusatullah/adt-complete-backup-rbma

ADT RESTORE STATUS
Files restored: 2744
Existing files preserved: 0
Repository: /kaggle/working/ADT_CV/ADT

RBMA13 learned-PCEN checkpoint status:
Fold 1: COMPLETE
Fold 2: COMPLETE
Fold 3: NOT COMPLETE

RESTORE CHECK FINISHED
Existing experiments were not overwritten.


In [2]:
%pip install -q partitura

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.9/363.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.6/504.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.9/261.9 kB 11.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:

from pathlib import Path
import subprocess
import sys
import json

# ============================================================
# RBMA13 — LEARNABLE PCEN — FOLD 3 ONLY
# MEL SPECTROGRAM GENERATION
# ============================================================

WORK = Path("/kaggle/working")
REPO = WORK / "ADT_CV" / "ADT"
INPUT = Path("/kaggle/input")
PREP = REPO / "io/prepare_vogl_cv.py"

assert PREP.is_file()

# Locate the attached RBMA13 audio dataset.
RBMA_ROOT = Path(
    "/kaggle/input/datasets/wusatullah/adt-rbma/RBMA"
)

if not (
    (RBMA_ROOT / "annotations").is_dir()
    and len(list(RBMA_ROOT.glob("*.wav"))) >= 27
):
    candidates = [
        p.parent
        for p in INPUT.rglob("annotations")
        if p.is_dir()
        and len(list(p.parent.glob("*.wav"))) >= 27
    ]
    assert len(candidates) == 1, (
        f"Cannot uniquely locate RBMA13: {candidates}"
    )
    RBMA_ROOT = candidates[0]

print("RBMA13 audio root:", RBMA_ROOT)

DATA_ROOT = REPO / "data/VoglCV"
FOLD_DIR = DATA_ROOT / "rbma13/fold_3"

command = [
    sys.executable, "-u", str(PREP),
    "--dataset", "rbma13",
    "--representation", "mel",
    "--fold", "3",
    "--rbma-root", str(RBMA_ROOT),
    "--output-root", str(DATA_ROOT),
    "--seed", "42",
    "--validation-fraction", "0.15"
]

subprocess.run(command, cwd=REPO, check=True)

# Verify the recording-level Mel inputs.
expected_counts = {
    "train": 15,
    "validation": 3,
    "test": 9
}

for split, expected in expected_counts.items():
    folder = (
        FOLD_DIR / "mel_recordings"
        / split / "rbma13"
    )

    tensors = list(folder.glob("*.pt"))

    assert len(tensors) == expected, (
        f"{split}: expected {expected}, "
        f"found {len(tensors)}"
    )

    assert all(p.stat().st_size > 0 for p in tensors)
    print(f"{split}: {len(tensors)} recordings verified")

manifest = FOLD_DIR / "split_manifest.json"
assert manifest.is_file()

info = json.loads(manifest.read_text())
assert info["dataset"] == "rbma13"
assert info["fold"] == 3

print("\nRBMA13 FOLD 3 MEL SPECTROGRAMS READY")


RBMA13 audio root: /kaggle/input/datasets/wusatullah/adt-rbma/RBMA
Dataset: rbma13, 27 annotated recordings, 1 folds
Fold 3 {'train': 15, 'validation': 3, 'test': 9}
rbma13 fold 3 train mel 1/15
rbma13 fold 3 train mel 2/15
rbma13 fold 3 train mel 3/15
rbma13 fold 3 train mel 4/15
rbma13 fold 3 train mel 5/15
rbma13 fold 3 train mel 6/15
rbma13 fold 3 train mel 7/15
rbma13 fold 3 train mel 8/15
rbma13 fold 3 train mel 9/15
rbma13 fold 3 train mel 10/15
rbma13 fold 3 train mel 11/15
rbma13 fold 3 train mel 12/15
rbma13 fold 3 train mel 13/15
rbma13 fold 3 train mel 14/15
rbma13 fold 3 train mel 15/15
rbma13 fold 3 validation mel 1/3
rbma13 fold 3 validation mel 2/3
rbma13 fold 3 validation mel 3/3
rbma13 fold 3 test mel 1/9
rbma13 fold 3 test mel 2/9
rbma13 fold 3 test mel 3/9
rbma13 fold 3 test mel 4/9
rbma13 fold 3 test mel 5/9
rbma13 fold 3 test mel 6/9
rbma13 fold 3 test mel 7/9
rbma13 fold 3 test mel 8/9
rbma13 fold 3 test mel 9/9
train: 15 recordings verified
validation: 3 recordi

In [4]:

from pathlib import Path
from datetime import datetime
from IPython.display import display, FileLink
import subprocess
import sys
import json
import zipfile
import hashlib
import torch

# ============================================================
# SHARED TRAINING + BACKUP FUNCTIONS
# RBMA13 FOLD 3 TRAINING
# ============================================================

WORK = Path("/kaggle/working")
REPO = WORK / "ADT_CV" / "ADT"
BACKUP_DIR = WORK / "ADT_TRAINING_BACKUPS"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), (
    "Enable GPU acceleration in Kaggle."
)

def experiment_folder(dataset, representation, fold):
    return (
        REPO / "experiments/vogl_cnn_b"
        / representation
        / "Architecture"
        / f"{dataset}_{representation}"
        / dataset.upper()
        / f"fold_{fold}"
    )


def backup_completed_fold(
    dataset, representation, fold,
    training_command=None, log_file=None
):
    """
    Save a complete fold checkpoint and results ZIP.
    No raw audio or spectrogram tensors are included.
    """
    folder = experiment_folder(
        dataset, representation, fold
    )

    required = [
        "model.pt",
        "experiment_config.json",
        "test_results.json"
    ]

    if representation == "trainable_pcen":
        required.append("pcen.pt")

    for filename in required:
        path = folder / filename
        assert path.is_file() and path.stat().st_size > 0, (
            f"Missing checkpoint/result: {path}"
        )

    files = {}

    # Complete trained fold directory.
    for path in folder.rglob("*"):
        if path.is_file():
            files[path.relative_to(WORK).as_posix()] = path

    # Split manifest and fixed-PCEN metadata.
    data_dir = (
        REPO / "data/VoglCV"
        / dataset / f"fold_{fold}"
    )

    for filename in (
        "split_manifest.json",
        f"{dataset}_{representation}_parameters.json"
    ):
        path = data_dir / filename

        if path.is_file():
            files[path.relative_to(WORK).as_posix()] = path

    # Fixed-PCEN JSON configuration.
    params_file = (
        REPO / "experiments/pcen_ablation_config"
        / f"{dataset}_{representation}.json"
    )

    if params_file.is_file():
        files[
            params_file.relative_to(WORK).as_posix()
        ] = params_file

    # Training log.
    if log_file is not None:
        log_file = Path(log_file)
        if log_file.is_file():
            files[
                f"training_logs/{log_file.name}"
            ] = log_file

    # Relevant original repository scripts.
    for rel in (
        "run.py",
        "run_idmt_pcen_s_only.py",
        "io/prepare_vogl_cv.py",
        "models/trainable_pcen.py"
    ):
        path = REPO / rel
        if path.is_file():
            files[path.relative_to(WORK).as_posix()] = path

    results = json.loads(
        (folder / "test_results.json").read_text()
    )

    stamp = datetime.now().strftime(
        "%Y%m%d_%H%M%S_%f"
    )

    archive = BACKUP_DIR / (
        f"{dataset.upper()}_"
        f"{representation.upper()}_"
        f"FOLD{fold}_{stamp}.zip"
    )

    metadata = {
        "dataset": dataset,
        "representation": representation,
        "fold": fold,
        "model": "vogl_cnn_b",
        "test_micro_f1": float(
            results["micro"]["f1"]
        ),
        "training_command": training_command,
        "created_at": stamp,
        "files": sorted(files.keys())
    }

    with zipfile.ZipFile(
        archive, "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=4,
        allowZip64=True
    ) as zf:

        for name, path in sorted(files.items()):
            zf.write(path, arcname=name)

        zf.writestr(
            "FOLD_BACKUP_MANIFEST.json",
            json.dumps(metadata, indent=2)
        )

    with zipfile.ZipFile(archive) as zf:
        assert zf.testzip() is None, (
            f"ZIP verification failed: {archive}"
        )

    sha = hashlib.sha256()

    with archive.open("rb") as f:
        for block in iter(
            lambda: f.read(8 * 1024 * 1024), b""
        ):
            sha.update(block)

    checksum = archive.with_suffix(".zip.sha256")
    checksum.write_text(
        f"{sha.hexdigest()}  {archive.name}\n"
    )

    print("\n" + "=" * 70)
    print("BACKUP CREATED AND VERIFIED")
    print("=" * 70)
    print("ZIP:", archive)
    print(
        f"Size: {archive.stat().st_size / 1024**2:.2f} MB"
    )
    print("SHA256:", checksum)

    display(FileLink(str(archive)))

    return archive


def train_and_backup(dataset, representation, fold, runner):

    runner = Path(runner)
    assert runner.is_file(), f"Missing runner: {runner}"

    folder = experiment_folder(
        dataset, representation, fold
    )

    required = [
        folder / "model.pt",
        folder / "experiment_config.json",
        folder / "test_results.json"
    ]

    if representation == "trainable_pcen":
        required.append(folder / "pcen.pt")

    command = [
        sys.executable, "-u", str(runner),
        "--model", "vogl_cnn_b",
        "--dataset", dataset,
        "--fold", str(fold),
        "--representation", representation,
        "--num-samples", "1",
        "--batch-size", "8",
        "--cnn-frame-chunk", "8",
        "--num-epochs", "50",
        "--early-stop", "15",
        "--onset-tolerance-frames", "2",
        "--seed", "42"
    ]

    already_complete = all(
        p.is_file() and p.stat().st_size > 0
        for p in required
    )

    log_file = None

    if already_complete:
        print(
            f"{dataset} {representation} fold {fold}: "
            "already trained — skipping training."
        )
    else:
        if folder.exists() and any(folder.iterdir()):
            raise RuntimeError(
                f"Partial result folder exists: {folder}\n"
                "Inspect it before restarting."
            )

        stamp = datetime.now().strftime(
            "%Y%m%d_%H%M%S_%f"
        )

        log_file = (
            WORK
            / f"{dataset}_{representation}_"
              f"fold{fold}_{stamp}.log"
        )

        print("\n" + "=" * 70, flush=True)
        print(
            f"TRAINING {dataset.upper()} — "
            f"{representation} — FOLD {fold}",
            flush=True
        )
        print("=" * 70, flush=True)
        print("Log:", log_file, flush=True)

        with log_file.open(
            "w", encoding="utf-8"
        ) as logfile:

            process = subprocess.Popen(
                command,
                cwd=REPO,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1
            )

            for line in process.stdout:
                print(line, end="", flush=True)
                logfile.write(line)
                logfile.flush()

            return_code = process.wait()

        if return_code != 0:
            raise RuntimeError(
                f"Training failed: {dataset} "
                f"{representation} fold {fold}\n"
                f"See {log_file}"
            )

    # Verify all saved model files.
    for path in required:
        assert path.is_file() and path.stat().st_size > 0, (
            f"Missing result: {path}"
        )

    config = json.loads(
        (folder / "experiment_config.json").read_text()
    )

    assert (
        config.get("model") == "vogl_cnn_b"
        and config.get("representation") == representation
        and config.get("datasets") == [dataset]
        and config.get("cv_fold") == fold
    ), "Experiment metadata mismatch"

    results = json.loads(
        (folder / "test_results.json").read_text()
    )

    score = 100 * float(results["micro"]["f1"])

    print(
        f"\n{dataset.upper()} FOLD {fold} "
        f"TEST MICRO F1: {score:.2f}%"
    )

    # BACKUP IMMEDIATELY AFTER THIS FOLD.
    archive = backup_completed_fold(
        dataset=dataset,
        representation=representation,
        fold=fold,
        training_command=command,
        log_file=log_file
    )

    return score, archive


# ============================================================
# RUN RBMA13 LEARNABLE PCEN — FOLD 3
# ============================================================

DATA_DIR = REPO / "data/VoglCV/rbma13/fold_3"

for split in ("train", "validation", "test"):
    folder = (
        DATA_DIR / "mel_recordings"
        / split / "rbma13"
    )

    assert folder.is_dir() and any(folder.glob("*.pt")), (
        f"Missing RBMA Mel spectrograms: {folder}"
    )

RBMA_SCORE, RBMA_ZIP = train_and_backup(
    dataset="rbma13",
    representation="trainable_pcen",
    fold=3,
    runner=REPO / "run.py"
)

print("\nRBMA13 LEARNABLE PCEN FOLD 3 COMPLETE")
print("Backup:", RBMA_ZIP)



TRAINING RBMA13 — trainable_pcen — FOLD 3
Log: /kaggle/working/rbma13_trainable_pcen_fold3_20261113_104724_297966.log
Dataset files:
  Train: /kaggle/working/ADT_CV/ADT/data/VoglCV/rbma13/fold_3
  Validation: /kaggle/working/ADT_CV/ADT/data/VoglCV/rbma13/fold_3
  Test: /kaggle/working/ADT_CV/ADT/data/VoglCV/rbma13/fold_3

Using device: cuda
Model: vogl_cnn_b
Representation: trainable_pcen
Dataset: ['rbma13']
Study: Architecture
Experiment: rbma13_trainable_pcen
\nTraining dataset size: 15
Computing training-only normalization statistics...
Training mean: 0.15376198291778564
Training std: 0.21552498638629913

Initializing Ray...
2026-11-13 10:47:44,961	INFO worker.py:2024 -- Started a local Ray instance.

Starting training...
2026-11-13 10:47:45,796	INFO tensorboardx.py:43 -- pip install "ray[tune]" to see TensorBoard files.
2026-11-13 10:47:45,796	WARNING callback.py:132 -- The TensorboardX logger cannot be instantiated because either TensorboardX or one of it's dependencies is not in

/kaggle/working/ADT_TRAINING_BACKUPS/RBMA13_TRAINABLE_PCEN_FOLD3_20261113_154327_324532.zip


RBMA13 LEARNABLE PCEN FOLD 3 COMPLETE
Backup: /kaggle/working/ADT_TRAINING_BACKUPS/RBMA13_TRAINABLE_PCEN_FOLD3_20261113_154327_324532.zip


In [5]:

from pathlib import Path
import subprocess
import sys
import json
import hashlib
import shutil
import os

# ============================================================
# ENST ACCOMPANIED — EXPERIMENT 2
# FIXED PCEN SMOOTHING ONLY
# SPECTROGRAM GENERATION — FOLDS 1, 2, 3
# ============================================================

REPO = Path("/kaggle/working/ADT_CV/ADT")
INPUT = Path("/kaggle/input")
PREP = REPO / "io/prepare_vogl_cv.py"

DATASET = "enst_accompanied"
REP = "pcen_s_only"

PARAMS = {
    "s": 0.018356,
    "alpha": 0.98,
    "delta": 2.0,
    "r": 0.5
}

assert PREP.is_file()

# ------------------------------------------------------------
# LOCATE ENST AUDIO
# ------------------------------------------------------------

DATA_ROOT = Path(
    "/kaggle/input/datasets/wusatullah/"
    "adt-datasets/ADT_Datasets"
)

ENST_DIR = DATA_ROOT / "ENST-drums-public"

if not (
    ENST_DIR / "drummer_1/audio/accompaniment"
).is_dir():

    candidates = [
        p for p in INPUT.rglob("ENST-drums-public")
        if p.is_dir()
        and (p / "drummer_1/audio/accompaniment").is_dir()
    ]

    assert len(candidates) == 1, (
        f"Cannot uniquely locate ENST audio: {candidates}"
    )

    ENST_DIR = candidates[0]
    DATA_ROOT = ENST_DIR.parent

print("ENST audio root:", ENST_DIR)
print("PCEN parameters:", PARAMS)

# ------------------------------------------------------------
# PARAMETER CONFIGURATION
# ------------------------------------------------------------

CONFIG = (
    REPO / "experiments/pcen_ablation_config"
    / "enst_accompanied_pcen_s_only.json"
)

CONFIG.parent.mkdir(parents=True, exist_ok=True)

if CONFIG.exists():
    old = json.loads(CONFIG.read_text())
    old = old.get("parameters", old)

    assert old == PARAMS, (
        f"Existing parameter configuration differs: {CONFIG}"
    )
else:
    CONFIG.write_text(
        json.dumps(PARAMS, indent=2),
        encoding="utf-8"
    )

fingerprint = hashlib.sha256(
    json.dumps(PARAMS, sort_keys=True).encode()
).hexdigest()[:12]

STAGING = (
    REPO / "data/VoglCV"
    / f"enst_accompanied_s_only_staging_{fingerprint}"
)

DEST_ROOT = (
    REPO / "data/VoglCV/enst_accompanied"
)

def file_hash(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(8 * 1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()

# ------------------------------------------------------------
# GENERATE THREE FOLDS
# ------------------------------------------------------------

for fold in (1, 2, 3):

    print("\n" + "=" * 70, flush=True)
    print(
        f"ENST ACCOMPANIED EXP 2 — "
        f"SPECTROGRAM FOLD {fold}",
        flush=True
    )
    print("=" * 70, flush=True)

    dest = DEST_ROOT / f"fold_{fold}"
    dest.mkdir(parents=True, exist_ok=True)

    metadata_file = (
        dest / f"{DATASET}_{REP}_parameters.json"
    )

    expected = [
        dest / f"{DATASET}_{REP}_{split}.pt"
        for split in ("train", "validation", "test")
    ]

    if metadata_file.is_file():
        old = json.loads(metadata_file.read_text())
        assert old["parameters"] == PARAMS, (
            f"Fold {fold}: PCEN metadata mismatch"
        )

        if all(
            p.is_file() and p.stat().st_size > 0
            for p in expected
        ):
            print("Already generated — skipping.")
            continue

    elif any(p.exists() for p in expected):
        raise RuntimeError(
            f"Existing features without metadata: {dest}"
        )

    command = [
        sys.executable, "-u", str(PREP),
        "--dataset", DATASET,
        "--representation", "optimized_pcen",
        "--fold", str(fold),
        "--pcen-params", str(CONFIG),
        "--data-root", str(DATA_ROOT),
        "--output-root", str(STAGING),
        "--seed", "42",
        "--validation-fraction", "0.15"
    ]

    subprocess.run(
        command,
        cwd=REPO,
        check=True
    )

    generated = (
        STAGING / DATASET / f"fold_{fold}"
    )

    # Preserve original CV split assignments.
    new_manifest = generated / "split_manifest.json"
    old_manifest = dest / "split_manifest.json"

    assert new_manifest.is_file()

    if old_manifest.exists():
        assert (
            json.loads(old_manifest.read_text())
            == json.loads(new_manifest.read_text())
        ), f"Fold {fold}: CV split mismatch!"
    else:
        shutil.copy2(new_manifest, old_manifest)

    # Copy under the representation expected by CNN-b.
    for split in ("train", "validation", "test"):

        source = (
            generated
            / f"{DATASET}_optimized_pcen_{split}.pt"
        )

        destination = (
            dest / f"{DATASET}_{REP}_{split}.pt"
        )

        assert (
            source.is_file()
            and source.stat().st_size > 0
        ), f"Missing generated tensor: {source}"

        if destination.exists():
            assert file_hash(source) == file_hash(destination), (
                f"Existing tensor differs: {destination}"
            )
            print("Verified:", destination.name)
            continue

        temporary = destination.with_name(
            destination.name + ".copying"
        )

        try:
            shutil.copy2(source, temporary)
            os.replace(temporary, destination)
        finally:
            temporary.unlink(missing_ok=True)

        print(
            f"Saved: {destination.name} "
            f"({destination.stat().st_size / 1024**2:.1f} MB)",
            flush=True
        )

    metadata_file.write_text(
        json.dumps({
            "dataset": DATASET,
            "representation": REP,
            "parameters": PARAMS,
            "fold": fold,
            "seed": 42,
            "validation_fraction": 0.15
        }, indent=2),
        encoding="utf-8"
    )

    print(f"Fold {fold}: COMPLETE", flush=True)

# ------------------------------------------------------------
# VERIFY ALL NINE TENSORS
# ------------------------------------------------------------

count = 0

for fold in (1, 2, 3):
    dest = DEST_ROOT / f"fold_{fold}"

    for split in ("train", "validation", "test"):

        path = (
            dest / f"{DATASET}_{REP}_{split}.pt"
        )

        assert path.is_file() and path.stat().st_size > 0
        count += 1

print("\n" + "=" * 70)
print(f"ENST ACCOMPANIED EXPERIMENT 2: {count}/9 TENSORS READY")
print("=" * 70)


ENST audio root: /kaggle/input/datasets/wusatullah/adt-datasets/ADT_Datasets/ENST-drums-public
PCEN parameters: {'s': 0.018356, 'alpha': 0.98, 'delta': 2.0, 'r': 0.5}

ENST ACCOMPANIED EXP 2 — SPECTROGRAM FOLD 1
ENST matched recordings: 64
Unmatched accompaniment files: 0
Dataset: enst_accompanied, 64 annotated recordings, 1 folds
Fold 1 {'train': 37, 'validation': 6, 'test': 21}
enst_accompanied fold 1 train optimized_pcen 1/37
enst_accompanied fold 1 train optimized_pcen 2/37
enst_accompanied fold 1 train optimized_pcen 3/37
enst_accompanied fold 1 train optimized_pcen 4/37
enst_accompanied fold 1 train optimized_pcen 5/37
enst_accompanied fold 1 train optimized_pcen 6/37
enst_accompanied fold 1 train optimized_pcen 7/37
enst_accompanied fold 1 train optimized_pcen 8/37
enst_accompanied fold 1 train optimized_pcen 9/37
enst_accompanied fold 1 train optimized_pcen 10/37
enst_accompanied fold 1 train optimized_pcen 11/37
enst_accompanied fold 1 train optimized_pcen 12/37
enst_accompani

In [6]:

from pathlib import Path
from datetime import datetime
from IPython.display import display, FileLink
import csv
import json
import zipfile
import statistics

# ============================================================
# ENST ACCOMPANIED — EXPERIMENT 2
# TRAIN FOLDS 1, 2, 3
# BACKUP IMMEDIATELY AFTER EACH FOLD
# ============================================================

REPO = Path("/kaggle/working/ADT_CV/ADT")
WORK = Path("/kaggle/working")
BACKUP_DIR = WORK / "ADT_TRAINING_BACKUPS"

DATASET = "enst_accompanied"
REP = "pcen_s_only"

RUNNER = REPO / "run_idmt_pcen_s_only.py"
assert RUNNER.is_file()
assert callable(train_and_backup)

PARAMS = {
    "s": 0.018356,
    "alpha": 0.98,
    "delta": 2.0,
    "r": 0.5
}

# ------------------------------------------------------------
# VERIFY ALL FEATURE TENSORS AND PARAMETER METADATA
# ------------------------------------------------------------

for fold in (1, 2, 3):

    folder = (
        REPO / "data/VoglCV"
        / DATASET / f"fold_{fold}"
    )

    metadata_path = (
        folder / f"{DATASET}_{REP}_parameters.json"
    )

    assert metadata_path.is_file()
    metadata = json.loads(metadata_path.read_text())

    assert metadata["parameters"] == PARAMS, (
        f"Fold {fold}: wrong PCEN parameters!"
    )

    for split in ("train", "validation", "test"):

        path = (
            folder / f"{DATASET}_{REP}_{split}.pt"
        )

        assert path.is_file() and path.stat().st_size > 0, (
            f"Missing: {path}"
        )

print("All ENST Accompanied Experiment 2 features verified.")

# ------------------------------------------------------------
# TRAIN + BACK UP EACH COMPLETED FOLD
# ------------------------------------------------------------

scores = []
backups = []

for fold in (1, 2, 3):

    print("\n" + "=" * 75, flush=True)
    print(
        f"ENST ACCOMPANIED EXPERIMENT 2 — "
        f"TRAINING FOLD {fold}",
        flush=True
    )
    print("=" * 75, flush=True)

    score, backup_zip = train_and_backup(
        dataset=DATASET,
        representation=REP,
        fold=fold,
        runner=RUNNER
    )

    scores.append(score)
    backups.append(backup_zip)

    print(
        f"\nFOLD {fold} FINISHED AND BACKED UP",
        flush=True
    )

# ------------------------------------------------------------
# SAVE THREE-FOLD RESULT CSV
# ------------------------------------------------------------

csv_path = (
    BACKUP_DIR
    / "ENST_ACCOMPANIED_EXP2_SMOOTHING_ONLY_RESULTS.csv"
)

rows = []

for fold in (1, 2, 3):

    result_dir = experiment_folder(
        DATASET, REP, fold
    )

    results = json.loads(
        (result_dir / "test_results.json").read_text()
    )

    micro = results["micro"]
    classes = results["classes"]

    rows.append({
        "Dataset": DATASET,
        "Experiment": 2,
        "Representation": REP,
        "Fold": fold,
        "Precision (%)": 100 * micro["precision"],
        "Recall (%)": 100 * micro["recall"],
        "Micro F1 (%)": 100 * micro["f1"],
        "Kick F1 (%)": 100 * classes["bass_drum"]["f1"],
        "Snare F1 (%)": 100 * classes["snare_drum"]["f1"],
        "Hi-Hat F1 (%)": 100 * classes["hi_hat"]["f1"],
    })

with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f, fieldnames=list(rows[0].keys())
    )
    writer.writeheader()
    writer.writerows(rows)

# ------------------------------------------------------------
# CREATE ADDITIONAL COMBINED EXPERIMENT BACKUP
# ------------------------------------------------------------

stamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S_%f"
)

combined_zip = (
    BACKUP_DIR
    / f"ENST_ACCOMPANIED_EXP2_ALL_3_FOLDS_{stamp}.zip"
)

files = {}

for fold in (1, 2, 3):

    result_dir = experiment_folder(
        DATASET, REP, fold
    )

    for path in result_dir.rglob("*"):
        if path.is_file():
            files[path.relative_to(WORK).as_posix()] = path

    data_dir = (
        REPO / "data/VoglCV"
        / DATASET / f"fold_{fold}"
    )

    for name in (
        "split_manifest.json",
        f"{DATASET}_{REP}_parameters.json"
    ):
        path = data_dir / name

        if path.is_file():
            files[path.relative_to(WORK).as_posix()] = path

config_path = (
    REPO / "experiments/pcen_ablation_config"
    / f"{DATASET}_{REP}.json"
)

if config_path.is_file():
    files[config_path.relative_to(WORK).as_posix()] = (
        config_path
    )

for log in WORK.glob(
    "enst_accompanied_pcen_s_only_fold*.log"
):
    files[f"training_logs/{log.name}"] = log

files[csv_path.name] = csv_path

with zipfile.ZipFile(
    combined_zip, "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=4,
    allowZip64=True
) as zf:

    for name, path in sorted(files.items()):
        zf.write(path, arcname=name)

    zf.writestr(
        "EXPERIMENT_BACKUP_MANIFEST.json",
        json.dumps({
            "dataset": DATASET,
            "experiment": 2,
            "representation": REP,
            "parameters": PARAMS,
            "fold_micro_f1_percent": scores,
            "mean_micro_f1_percent": statistics.mean(scores),
            "sd_micro_f1_pp": statistics.stdev(scores),
            "created_at": stamp
        }, indent=2)
    )

with zipfile.ZipFile(combined_zip) as zf:
    assert zf.testzip() is None

print("\n" + "=" * 75)
print("ENST ACCOMPANIED EXPERIMENT 2 COMPLETE")
print("=" * 75)

for fold, score in enumerate(scores, start=1):
    print(f"Fold {fold}: {score:.2f}%")

print(
    f"Mean Test Micro F1: "
    f"{statistics.mean(scores):.2f}%"
)
print(
    f"Standard Deviation: "
    f"{statistics.stdev(scores):.2f} pp"
)

print("\nIndividual fold backups:")
for path in backups:
    print(path.name)

print("\nCombined experiment backup:", combined_zip)
print("Results CSV:", csv_path)

display(FileLink(str(combined_zip)))
display(FileLink(str(csv_path)))


All ENST Accompanied Experiment 2 features verified.

ENST ACCOMPANIED EXPERIMENT 2 — TRAINING FOLD 1

TRAINING ENST_ACCOMPANIED — pcen_s_only — FOLD 1
Log: /kaggle/working/enst_accompanied_pcen_s_only_fold1_20261113_154436_157573.log
Dataset files:
  Train: /kaggle/working/ADT_CV/ADT/data/VoglCV/enst_accompanied/fold_1/enst_accompanied_pcen_s_only_train.pt
  Validation: /kaggle/working/ADT_CV/ADT/data/VoglCV/enst_accompanied/fold_1/enst_accompanied_pcen_s_only_validation.pt
  Test: /kaggle/working/ADT_CV/ADT/data/VoglCV/enst_accompanied/fold_1/enst_accompanied_pcen_s_only_test.pt

Using device: cuda
Model: vogl_cnn_b
Representation: pcen_s_only
Dataset: ['enst_accompanied']
Study: Architecture
Experiment: enst_accompanied_pcen_s_only
\nTraining dataset size: 547
Computing training-only normalization statistics...
Calculating chunk-based normalization...
  Normalized statistics: 200/547 chunks
  Normalized statistics: 400/547 chunks
Normalization completed for 547 chunks.
Training mean

/kaggle/working/ADT_TRAINING_BACKUPS/ENST_ACCOMPANIED_PCEN_S_ONLY_FOLD1_20261113_162040_015298.zip


FOLD 1 FINISHED AND BACKED UP

ENST ACCOMPANIED EXPERIMENT 2 — TRAINING FOLD 2

TRAINING ENST_ACCOMPANIED — pcen_s_only — FOLD 2
Log: /kaggle/working/enst_accompanied_pcen_s_only_fold2_20261113_162040_203221.log
Dataset files:
  Train: /kaggle/working/ADT_CV/ADT/data/VoglCV/enst_accompanied/fold_2/enst_accompanied_pcen_s_only_train.pt
  Validation: /kaggle/working/ADT_CV/ADT/data/VoglCV/enst_accompanied/fold_2/enst_accompanied_pcen_s_only_validation.pt
  Test: /kaggle/working/ADT_CV/ADT/data/VoglCV/enst_accompanied/fold_2/enst_accompanied_pcen_s_only_test.pt

Using device: cuda
Model: vogl_cnn_b
Representation: pcen_s_only
Dataset: ['enst_accompanied']
Study: Architecture
Experiment: enst_accompanied_pcen_s_only
\nTraining dataset size: 510
Computing training-only normalization statistics...
Calculating chunk-based normalization...
  Normalized statistics: 200/510 chunks
  Normalized statistics: 400/510 chunks
Normalization completed for 510 chunks.
Training mean: 0.1516973078250885
T

/kaggle/working/ADT_TRAINING_BACKUPS/ENST_ACCOMPANIED_PCEN_S_ONLY_FOLD2_20261113_165205_194359.zip


FOLD 2 FINISHED AND BACKED UP

ENST ACCOMPANIED EXPERIMENT 2 — TRAINING FOLD 3

TRAINING ENST_ACCOMPANIED — pcen_s_only — FOLD 3
Log: /kaggle/working/enst_accompanied_pcen_s_only_fold3_20261113_165205_383396.log
Dataset files:
  Train: /kaggle/working/ADT_CV/ADT/data/VoglCV/enst_accompanied/fold_3/enst_accompanied_pcen_s_only_train.pt
  Validation: /kaggle/working/ADT_CV/ADT/data/VoglCV/enst_accompanied/fold_3/enst_accompanied_pcen_s_only_validation.pt
  Test: /kaggle/working/ADT_CV/ADT/data/VoglCV/enst_accompanied/fold_3/enst_accompanied_pcen_s_only_test.pt

Using device: cuda
Model: vogl_cnn_b
Representation: pcen_s_only
Dataset: ['enst_accompanied']
Study: Architecture
Experiment: enst_accompanied_pcen_s_only
\nTraining dataset size: 543
Computing training-only normalization statistics...
Calculating chunk-based normalization...
  Normalized statistics: 200/543 chunks
  Normalized statistics: 400/543 chunks
Normalization completed for 543 chunks.
Training mean: 0.1505156010389328
T

/kaggle/working/ADT_TRAINING_BACKUPS/ENST_ACCOMPANIED_PCEN_S_ONLY_FOLD3_20261113_172701_729297.zip


FOLD 3 FINISHED AND BACKED UP

ENST ACCOMPANIED EXPERIMENT 2 COMPLETE
Fold 1: 59.55%
Fold 2: 73.20%
Fold 3: 76.39%
Mean Test Micro F1: 69.71%
Standard Deviation: 8.95 pp

Individual fold backups:
ENST_ACCOMPANIED_PCEN_S_ONLY_FOLD1_20261113_162040_015298.zip
ENST_ACCOMPANIED_PCEN_S_ONLY_FOLD2_20261113_165205_194359.zip
ENST_ACCOMPANIED_PCEN_S_ONLY_FOLD3_20261113_172701_729297.zip

Combined experiment backup: /kaggle/working/ADT_TRAINING_BACKUPS/ENST_ACCOMPANIED_EXP2_ALL_3_FOLDS_20261113_172701_917387.zip
Results CSV: /kaggle/working/ADT_TRAINING_BACKUPS/ENST_ACCOMPANIED_EXP2_SMOOTHING_ONLY_RESULTS.csv


/kaggle/working/ADT_TRAINING_BACKUPS/ENST_ACCOMPANIED_EXP2_ALL_3_FOLDS_20261113_172701_917387.zip

/kaggle/working/ADT_TRAINING_BACKUPS/ENST_ACCOMPANIED_EXP2_SMOOTHING_ONLY_RESULTS.csv